In [ ]:
# Example 1: compat re-exports — sklearn_compat.utils mirrors sklearn's utils layout
# across versions (type_of_target moved modules over sklearn releases; the shim tracks it).
import numpy as np
from sklearn_compat.utils.extmath import safe_sqr
from sklearn_compat.utils.multiclass import type_of_target

kind = type_of_target(np.array([0, 1, 1, 0]))
print(f"type_of_target([0,1,1,0]) = {kind}")
assert kind == "binary"

kind_mc = type_of_target(np.array([0, 1, 2]))
print(f"type_of_target([0,1,2]) = {kind_mc}")
assert kind_mc == "multiclass"

sq = safe_sqr(np.array([1, 2, 3]))
print(f"safe_sqr([1,2,3]) = {sq}")
assert (sq == np.array([1, 4, 9])).all()
print("compat re-exports OK")

In [ ]:
# Example 2: validate_data shim — compatibility wrapper for sklearn's validate_data / _validate_data.
import numpy as np
from sklearn.base import BaseEstimator
from sklearn_compat.utils.validation import validate_data

class DummyEstimator(BaseEstimator):
    def fit(self, X, y=None):
        X = validate_data(self, X, reset=True)
        self.n_features_in_ = X.shape[1]
        return self

X = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]])
est = DummyEstimator().fit(X)
assert est.n_features_in_ == 3
print(f"validate_data accepted X.shape={X.shape}; n_features_in_={est.n_features_in_}")


In [ ]:
# Example 3: tags shim — sklearn_compat.utils._tags exposes get_tags() across sklearn versions.
from sklearn.linear_model import LogisticRegression
from sklearn_compat.utils._tags import get_tags

est = LogisticRegression()
tags = get_tags(est)
print(f"got tags object of type: {type(tags).__name__}")

# tags should be a namespace-like object (dataclass in newer sklearn, dict in older).
# Just check that we got *something* back.
assert tags is not None, "expected non-None tags"
print("tags retrieved OK")


In [ ]:
# Example 4: check_array compat re-export — basic input validation.
import numpy as np
from sklearn_compat.utils.validation import check_array

X_list = [[1, 2], [3, 4], [5, 6]]
X_arr = check_array(X_list)
assert isinstance(X_arr, np.ndarray)
assert X_arr.shape == (3, 2)
print(f"check_array converted list -> ndarray shape={X_arr.shape} dtype={X_arr.dtype}")

# Should reject 1D when 2D is required.
try:
    check_array(np.array([1, 2, 3]))
    raised = False
except ValueError:
    raised = True
assert raised, "expected ValueError on 1D input"
print("check_array correctly rejected 1D input")


In [ ]:
# Example 5: end-to-end fit/predict using sklearn_compat helpers inside a custom estimator.
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn_compat.utils.validation import validate_data

class MajorityClassifier(ClassifierMixin, BaseEstimator):
    def fit(self, X, y):
        X = validate_data(self, X, reset=True)
        vals, counts = np.unique(y, return_counts=True)
        self.classes_ = vals
        self.majority_ = vals[np.argmax(counts)]
        return self

    def predict(self, X):
        X = validate_data(self, X, reset=False)
        return np.full(X.shape[0], self.majority_)

rng = np.random.default_rng(0)
X = rng.normal(size=(20, 4))
y = np.array([0] * 15 + [1] * 5)
clf = MajorityClassifier().fit(X, y)
preds = clf.predict(X)
assert preds.shape == (20,)
assert (preds == 0).all(), f"expected all majority-class (0); got {preds}"
print(f"MajorityClassifier predicted majority class {clf.majority_} for all {len(preds)} samples")
print("sklearn_compat end-to-end OK")
